# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Usaf007/flyrankai-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Feature Vector Construction**
We are building a clean, modeled feature vector using the mid-panel test month (March 2026). The vector joins the daily fact performance table with the static dimension table to extract raw signals.

We explicitly filter for rows where `ga4_data_available IS TRUE` to ensure traffic metrics are present, and we drop any remaining nulls to guarantee a clean matrix for modeling.

In [1]:
import duckdb
import pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"
fact_table = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"
dim_table = f"{rel}/dim_content.parquet"

query = f"""
    SELECT
        f.gsc_impressions,
        f.ga4_sessions,
        f.gsc_clicks,
        date_diff('day', d.content_created_date, f.report_date) AS content_age_days,
        d.word_count,
        CAST(CASE WHEN f.gsc_avg_position > 10 THEN 1 ELSE 0 END AS INTEGER) AS target_is_declining
    FROM read_parquet('{fact_table}') f
    JOIN read_parquet('{dim_table}') d ON f.content_hash_id = d.content_hash_id
    WHERE f.ga4_data_available IS TRUE
    LIMIT 50000
"""
df_features = con.execute(query).df().dropna()
df_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_impressions,ga4_sessions,gsc_clicks,content_age_days,word_count,target_is_declining
0,0,1,0,11,4272,0
1,0,1,0,7,4143,0
2,0,1,0,7,3959,0
3,0,1,0,7,3854,0
4,0,1,0,5,3881,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**Feature Breakdown & Availability**

*   **`gsc_impressions` (Numeric):** Daily search visibility. Handled by explicit GA4 availability filter. Available at decision moment (subject to standard 24-48h API lag).
*   **`ga4_sessions` (Numeric):** Daily web traffic. Missing values excluded. Available at decision moment.
*   **`gsc_clicks` (Numeric):** Daily search clicks. Missing values excluded. Available at decision moment.
*   **`content_age_days` (Numeric):** Calculated delta between publish date and report date. Always calculable. Available at decision moment.
*   **`word_count` (Numeric):** Static article length. Always present in dimension table. Available at decision moment.

In [2]:
display(df_features.isnull().sum().to_frame(name='null_count'))
display(df_features[['gsc_impressions', 'ga4_sessions', 'gsc_clicks', 'content_age_days', 'word_count']].describe().round(2))

,null_count
gsc_impressions,0
ga4_sessions,0
gsc_clicks,0
content_age_days,0
word_count,0
target_is_declining,0


,gsc_impressions,ga4_sessions,gsc_clicks,content_age_days,word_count
count,47764.00,47764.00,47764.00,47764.00,47764.0
mean,246.08,1.92,1.13,156.68,3300.53
std,486.92,2.72,3.14,110.80,1236.57
min,0.00,0.00,0.00,0.00,48.0
25%,22.00,1.00,0.00,51.00,2689.0
50%,102.00,1.00,0.00,137.00,3149.0
75%,272.00,2.00,1.00,212.00,3677.0
max,16059.00,127.00,226.00,453.00,9644.0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Hunting Same-Day Proxies**
Our target (`target_is_declining`) is defined as average position dropping below 10. We need to ensure none of our features are secretly proxying this label.

If a page drops off page 1, its clicks will instantly plummet to zero on that exact same day. By training a quick Random Forest and inspecting the feature importances, we can see if the model is overly reliant on `gsc_clicks`. If clicks completely dominate the decision tree, it confirms structural leakage: the model isn't predicting a drop; it is merely observing that the page has already lost its clicks.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X = df_features[['gsc_impressions', 'ga4_sessions', 'gsc_clicks', 'content_age_days', 'word_count']]
y = df_features['target_is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(random_state=42, n_estimators=50).fit(X_tr, y_tr)

importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance Weight': clf.feature_importances_
}).sort_values(by='Importance Weight', ascending=False)

display(importances)

,Feature,Importance Weight
0,gsc_impressions,0.343920
4,word_count,0.319722
3,content_age_days,0.250132
2,gsc_clicks,0.045083
1,ga4_sessions,0.041143


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

**Excluded Fields:**
*   **`health_score`:** A composite downstream metric computed by internal app logic.
*   **`priority_score`:** A ranking metric designed for the end-user dashboard.
*   **`action_type` (Product Flags):** System-generated recommendations.

**Why they are excluded:** These are the *outputs* of the existing FlyRank software, not raw environmental signals. If we feed the application's own calculated health scores into the model to predict content health, we create a circular loop where the model simply memorizes the existing product logic instead of learning actual search patterns.

In [4]:
columns_query = f"""
    SELECT *
    FROM read_parquet('{fact_table}')
    LIMIT 1
"""
df_all_cols = con.execute(columns_query).df()

# Output available warehouse columns that were intentionally dropped
dropped_cols = [col for col in df_all_cols.columns if col not in df_features.columns]
print(dropped_cols)

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.